# Compute experiment data processing

This notebook:
- Creates a subdirectory in `analysis_plots` for each `compute_*.csv` in `measurements`, wipes and recreates them each run
- Copies each CSV into its subdirectory and adds derived columns
- **e** columns (e-prefixed): bound between -1 and +1
- **i** columns (i-prefixed): bound by ±1.5×IREFP
- Plots and filters will be added back later.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import shutil

def round_to_n_sig_figs(x, n=5):
    """Round scalar to n significant figures. Keeps 0, NaN, Inf unchanged."""
    if pd.isna(x) or x == 0 or np.isinf(x):
        return x
    try:
        return round(x, n - 1 - int(np.floor(np.log10(abs(x)))))
    except (ValueError, OverflowError):
        return x

def df_round_sig_figs(df, n=5):
    """Round all float columns to n significant figures (avoids 9.99...e-09 etc.)."""
    out = df.copy()
    for col in out.columns:
        if out[col].dtype.kind == 'f':
            out[col] = out[col].apply(lambda x: round_to_n_sig_figs(x, n))
    return out

# Absolute paths: same locations regardless of cwd (only .ipynb files stay in analysis_plots)
cwd = Path(os.getcwd()).resolve()
if cwd.name == 'analysis_plots':
    analysis_plots_dir = cwd
    measurements_dir = cwd.parent / 'measurements'
else:
    analysis_plots_dir = cwd / 'analysis_plots'
    measurements_dir = cwd / 'measurements'
analysis_plots_dir = analysis_plots_dir.resolve()
measurements_dir = measurements_dir.resolve()

print(f"analysis_plots_dir: {analysis_plots_dir}")
print(f"measurements_dir: {measurements_dir}")

# Step 1: Remove ALL compute_* subdirectories in analysis_plots (only notebooks remain)
if analysis_plots_dir.exists():
    for item in list(analysis_plots_dir.iterdir()):
        if item.is_dir() and item.name.startswith('compute_'):
            try:
                shutil.rmtree(item)
                print(f"Removed: {item.name}")
            except OSError as e:
                print(f"Warning: could not remove {item.name}: {e}")

# Step 2: Find all compute_*.csv in measurements
compute_csvs = sorted(measurements_dir.glob('compute_*.csv'))
print(f"Found {len(compute_csvs)} compute CSV(s): {[f.name for f in compute_csvs]}")

# Columns: value / IREFP, 3 decimal places
divide_by_irefp = ['KGAIN1', 'KGAIN2', 'TRIM1', 'TRIM2']
# Columns: A*2/IREFP - 1
linear_irefp = ['X1', 'X2', 'F11', 'F12', 'IMEAS']

processed_paths = []
for csv_path in compute_csvs:
    csv_path = csv_path.resolve()
    subdir_name = csv_path.stem  # e.g. compute_20260115_161903
    subdir = analysis_plots_dir / subdir_name
    subdir.mkdir(parents=True, exist_ok=True)
    dest_csv = subdir / csv_path.name
    shutil.copy2(str(csv_path), str(dest_csv))
    df = pd.read_csv(str(dest_csv))

    if 'IREFP' not in df.columns:
        print(f"  Skip {csv_path.name}: no IREFP column")
        continue
    irefp = df['IREFP'].replace(0, np.nan)

    for col in divide_by_irefp:
        if col in df.columns:
            df['e' + col] = (df[col] / irefp).round(3)

    for col in linear_irefp:
        if col in df.columns:
            df['e' + col] = df[col] * 2 / irefp - 1

    # Clip all e* columns to [-1, 1]
    e_cols = [c for c in df.columns if c.startswith('e')]
    for c in e_cols:
        df[c] = df[c].clip(-1.0, 1.0)

    # New formulas (row-wise)
    df['eXhat'] = df['eX1'] * df['eF11'] + df['eX2'] * df['eF12']
    df['eY'] = df['eIMEAS'] - df['eXhat']
    df['eXhatplus'] = df['eY'] * df['eKGAIN1'] + df['eXhat']
    df['iXhatplus'] = (df['eXhatplus'] + 1) * df['IREFP'] / 2

    # iDeltaX: if PPG_state is ERASE then iXhatplus - X1, else X1 - iXhatplus; minimum 1e-10
    is_erase = (df['PPG_state'] == 'ERASE')
    df['iDeltaX'] = np.where(is_erase, df['iXhatplus'] - df['X1'], df['X1'] - df['iXhatplus'])
    df['iDeltaX'] = df['iDeltaX'].clip(lower=1e-10)

    df['iOut1'] = df['X1'] * df['TRIM1'] / df['iDeltaX']

    # Clip all e* columns to [-1, 1]
    e_cols = [c for c in df.columns if c.startswith('e')]
    for c in e_cols:
        df[c] = df[c].clip(-1.0, 1.0)

    # Clip all i* columns to ±1.5*IREFP (per row)
    i_cols = [c for c in df.columns if c.startswith('i')]
    irefp_15 = 1.5 * df['IREFP']
    for c in i_cols:
        df[c] = df[c].clip(-irefp_15, irefp_15)

    # Round all floats to 5 significant figures (fixes 9.999...e-09 -> 1e-8 etc.)
    df = df_round_sig_figs(df, n=5)

    df.to_csv(str(dest_csv), index=False)
    processed_paths.append(dest_csv)
    print(f"  Created {subdir_name}/ and saved modified {csv_path.name}")

print(f"\nDone. Processed {len(processed_paths)} CSV(s).")
print("New columns: eKGAIN1, eKGAIN2, eTRIM1, eTRIM2, eX1, eX2, eF11, eF12, eIMEAS, eXhat, eY, eXhatplus, iXhatplus, iDeltaX, iOut1")

analysis_plots_dir: C:\Columbia\AdamResearch\kalman_2025\analysis_plots
measurements_dir: C:\Columbia\AdamResearch\kalman_2025\measurements
Found 3 compute CSV(s): ['compute_20260115_161903.csv', 'compute_20260115_171219.csv', 'compute_20260115_174521.csv']
  Created compute_20260115_161903/ and saved modified compute_20260115_161903.csv
  Created compute_20260115_171219/ and saved modified compute_20260115_171219.csv
  Created compute_20260115_174521/ and saved modified compute_20260115_174521.csv

Done. Processed 3 CSV(s).
New columns: eKGAIN1, eKGAIN2, eTRIM1, eTRIM2, eX1, eX2, eF11, eF12, eIMEAS, eXhat, eY, eXhatplus, iXhatplus, iDeltaX, iOut1


In [2]:
# (Data filters removed — add back later)

In [3]:
# (Plot config removed — add back later)

In [4]:
# (Plot function removed — add back later)

In [5]:
# (Plot generation removed — add back later)